# Group by

Às vezes, queremos selecionar dados com base em grupos e entender os dados agregados no nível do grupo. Dessa forma, em muitas análises de dados, queremos agrupar registros que possuem alguma característica em comum e então calcular estatísticas ou agregações para cada grupo. Por exemplo, podemos querer calcular a média de vendas por estado, a soma de população por cidade ou o número de alunos por curso.

Embora o Pandas permita iterar linha por linha em um DataFrame, essa abordagem é geralmente muito lenta e não é considerada idiomática. Sendo assim, para resolver esse tipo de problema de forma eficiente, o Pandas fornece o método groupby(). <br>
A ideia do groupby() é dividir o DataFrame em subconjuntos com base em uma ou mais colunas (chaves de agrupamento). Depois que os dados são divididos, podemos aplicar operações em cada grupo separadamente, como soma, média, contagem, máximo ou funções personalizadas.

O funcionamento do groupby segue um padrão muito importante na análise de dados, chamado split-apply-combine (dividir-aplicar-combinar):

1) Split (Dividir) <br>
O DataFrame é dividido em vários grupos com base nos valores de uma ou mais colunas.

2) Apply (Aplicar) <br>
Uma função é aplicada independentemente a cada grupo. Essa função pode ser uma agregação (mean, sum, count), uma transformação ou até uma função personalizada.

3) Combine (Combinar) <br>
Os resultados de cada grupo são combinados novamente em uma nova estrutura (normalmente um DataFrame ou uma Series).

Esse padrão é extremamente poderoso porque permite realizar operações complexas em grandes conjuntos de dados de forma vetorizada e muito mais eficiente do que iterar manualmente sobre as linhas.

In [2]:
# Importando as bibliotecas
import numpy as np
import pandas as pd

#### 1 - Splitting

In [57]:
# Vamos dar uma olhada em um exemplo.
# Vamos novamente dar uma olhada em alguns dados do censo dos EUA 

df = pd.read_csv('datasets/census.csv')

# Agora, vamos excluir resumos em nível estadual (que têm valor de 40), ou seja, vamos manter aqueles que têm valor de 50.
df = df[df['SUMLEV'] == 50]
df.head()

,SUMLEV,REGION,DIVISION,STATE,COUNTY,STNAME,CTYNAME,CENSUS2010POP,ESTIMATESBASE2010,POPESTIMATE2010,...,RDOMESTICMIG2011,RDOMESTICMIG2012,RDOMESTICMIG2013,RDOMESTICMIG2014,RDOMESTICMIG2015,RNETMIG2011,RNETMIG2012,RNETMIG2013,RNETMIG2014,RNETMIG2015
1,50,3,6,1,1,Alabama,Autauga County,54571,54571,54660,...,7.242091,-2.915927,-3.012349,2.265971,-2.530799,7.606016,-2.626146,-2.722002,2.592270,-2.187333
2,50,3,6,1,3,Alabama,Baldwin County,182265,182265,183193,...,14.832960,17.647293,21.845705,19.243287,17.197872,15.844176,18.559627,22.727626,20.317142,18.293499
3,50,3,6,1,5,Alabama,Barbour County,27457,27457,27341,...,-4.728132,-2.500690,-7.056824,-3.904217,-10.543299,-4.874741,-2.758113,-7.167664,-3.978583,-10.543299
4,50,3,6,1,7,Alabama,Bibb County,22915,22919,22861,...,-5.527043,-5.068871,-6.201001,-0.177537,0.177258,-5.088389,-4.363636,-5.403729,0.754533,1.107861
5,50,3,6,1,9,Alabama,Blount County,57322,57322,57373,...,1.807375,-1.177622,-1.748766,-2.062535,-1.369970,1.859511,-0.848580,-1.402476,-1.577232,-0.884411


Para este exemplo de groupby(), utilizaremos o conjunto de dados do censo. O objetivo é calcular a população média dos condados (CENSUS2010POP) em cada estado (STNAME).

Antes de usar groupby(), vamos resolver o problema de forma manual, iterando sobre os estados. Isso servirá para mostrar por que essa abordagem não é a mais eficiente em Pandas.

Primeiro, obtemos uma lista com todos os estados únicos do DataFrame usando: df['STNAME'].unique(). Lembrando que isso retorna um array contendo cada estado apenas uma vez.

Vamos executar essa tarefa três vezes e cronometrá-la. Para isso, usaremos a magic function %%timeit.

In [ ]:
%%timeit -n 3

for state in df['STNAME'].unique():
    # Vamos apenas calcular a média usando o numpy para esse estado específico.
    media = np.average(df.loc[df['STNAME'] == state, 'CENSUS2010POP'])

    # Vamos imprimir na tela
    print(f'Condados do estado {state}\nPopulação média de {str(media)} ')

Condados em estado Alabama
População média de 71339.34328358209 
Condados em estado Alaska
População média de 24490.724137931036 
Condados em estado Arizona
População média de 426134.4666666667 
Condados em estado Arkansas
População média de 38878.90666666667 
Condados em estado California
População média de 642309.5862068966 
Condados em estado Colorado
População média de 78581.1875 
Condados em estado Connecticut
População média de 446762.125 
Condados em estado Delaware
População média de 299311.3333333333 
Condados em estado District of Columbia
População média de 601723.0 
Condados em estado Florida
População média de 280616.5671641791 
Condados em estado Georgia
População média de 60928.63522012578 
Condados em estado Hawaii
População média de 272060.2 
Condados em estado Idaho
População média de 35626.86363636364 
Condados em estado Illinois
População média de 125790.50980392157 
Condados em estado Indiana
População média de 70476.10869565218 
Condados em estado Iowa
População m

Agora, vamos tentar a abordagem usando groupby

In [ ]:
%%timeit -n 3

# Para esse método, começaremos dizendo ao Pandas que estamosinteressados em agrupar o estado pelo nome, uma vez que essa é a nossa divisão. 
for grupo, frame in df.groupby('STNAME'):
    # Perceba que temos duas variáveis (grupo e frame), isso porque o groupby retorna uma tupla em que o primeiro valor é o valor da chave que estamos tentando agrupar, que nesse caso, é um nome de estado específico, já o segundo é um DataFrame projetado que foi encontrado para esse grupo. Portanto, isso o groupby retorna uma tupla.

    # Agora podemos incluir nossa lógica de cálculo
    media = np.average(frame['CENSUS2010POP'])
    print(f'Condado do estado {grupo}\nPopulação média de {str(media)} ')

Condado do estado Alabama
População média de 71339.34328358209 
Condado do estado Alaska
População média de 24490.724137931036 
Condado do estado Arizona
População média de 426134.4666666667 
Condado do estado Arkansas
População média de 38878.90666666667 
Condado do estado California
População média de 642309.5862068966 
Condado do estado Colorado
População média de 78581.1875 
Condado do estado Connecticut
População média de 446762.125 
Condado do estado Delaware
População média de 299311.3333333333 
Condado do estado District of Columbia
População média de 601723.0 
Condado do estado Florida
População média de 280616.5671641791 
Condado do estado Georgia
População média de 60928.63522012578 
Condado do estado Hawaii
População média de 272060.2 
Condado do estado Idaho
População média de 35626.86363636364 
Condado do estado Illinois
População média de 125790.50980392157 
Condado do estado Indiana
População média de 70476.10869565218 
Condado do estado Iowa
População média de 30771.26

Perceba que a diferença é metade do tempo:
- Para o primeiro caso: 30.8 ms ± 3.49 ms per loop (mean ± std. dev. of 7 runs, 3 loops each)
- Usando groupby: 14.4 ms ± 2.83 ms per loop (mean ± std. dev. of 7 runs, 3 loops each)

In [38]:
# Um ponto importante, é que o groupby() não retorna um DataFrame, ele retorna um objeto especial chamado DataFrameGroupBy. Esse objeto funciona de forma lazy (preguiçosa) igual a função map(), uma vez que ele apenas define os grupos, mas não calcula nada ainda, com o objetivo de economizar memória. 

# Criando o groupby
grupos = df.groupby('STNAME')
print(f'Tipo do grupo: {type(grupos)}')

# Agora precisamos iterar sobre os grupos, ou utilizar o next(). Como já iteramos para calcular a média, vamos usar o next()
estado, frame = next(iter(grupos))

print(f'Estado: {estado}')
display(frame.head())

Tipo do grupo: <class 'pandas.api.typing.DataFrameGroupBy'>
Estado: Alabama


,SUMLEV,REGION,DIVISION,STATE,COUNTY,STNAME,CTYNAME,CENSUS2010POP,ESTIMATESBASE2010,POPESTIMATE2010,...,RDOMESTICMIG2011,RDOMESTICMIG2012,RDOMESTICMIG2013,RDOMESTICMIG2014,RDOMESTICMIG2015,RNETMIG2011,RNETMIG2012,RNETMIG2013,RNETMIG2014,RNETMIG2015
1,50,3,6,1,1,Alabama,Autauga County,54571,54571,54660,...,7.242091,-2.915927,-3.012349,2.265971,-2.530799,7.606016,-2.626146,-2.722002,2.592270,-2.187333
2,50,3,6,1,3,Alabama,Baldwin County,182265,182265,183193,...,14.832960,17.647293,21.845705,19.243287,17.197872,15.844176,18.559627,22.727626,20.317142,18.293499
3,50,3,6,1,5,Alabama,Barbour County,27457,27457,27341,...,-4.728132,-2.500690,-7.056824,-3.904217,-10.543299,-4.874741,-2.758113,-7.167664,-3.978583,-10.543299
4,50,3,6,1,7,Alabama,Bibb County,22915,22919,22861,...,-5.527043,-5.068871,-6.201001,-0.177537,0.177258,-5.088389,-4.363636,-5.403729,0.754533,1.107861
5,50,3,6,1,9,Alabama,Blount County,57322,57322,57373,...,1.807375,-1.177622,-1.748766,-2.062535,-1.369970,1.859511,-0.848580,-1.402476,-1.577232,-0.884411


Se analisar o frame inteiro, teremos apenas o estado do Alabama.

Também podemos acessar um frame direto pela chave, se essa for conhecida: 

In [40]:
# Acessar um grupo específico
df.groupby('STNAME').get_group('Alabama').head()

,SUMLEV,REGION,DIVISION,STATE,COUNTY,STNAME,CTYNAME,CENSUS2010POP,ESTIMATESBASE2010,POPESTIMATE2010,...,RDOMESTICMIG2011,RDOMESTICMIG2012,RDOMESTICMIG2013,RDOMESTICMIG2014,RDOMESTICMIG2015,RNETMIG2011,RNETMIG2012,RNETMIG2013,RNETMIG2014,RNETMIG2015
1,50,3,6,1,1,Alabama,Autauga County,54571,54571,54660,...,7.242091,-2.915927,-3.012349,2.265971,-2.530799,7.606016,-2.626146,-2.722002,2.592270,-2.187333
2,50,3,6,1,3,Alabama,Baldwin County,182265,182265,183193,...,14.832960,17.647293,21.845705,19.243287,17.197872,15.844176,18.559627,22.727626,20.317142,18.293499
3,50,3,6,1,5,Alabama,Barbour County,27457,27457,27341,...,-4.728132,-2.500690,-7.056824,-3.904217,-10.543299,-4.874741,-2.758113,-7.167664,-3.978583,-10.543299
4,50,3,6,1,7,Alabama,Bibb County,22915,22919,22861,...,-5.527043,-5.068871,-6.201001,-0.177537,0.177258,-5.088389,-4.363636,-5.403729,0.754533,1.107861
5,50,3,6,1,9,Alabama,Blount County,57322,57322,57373,...,1.807375,-1.177622,-1.748766,-2.062535,-1.369970,1.859511,-0.848580,-1.402476,-1.577232,-0.884411


In [44]:
# Na forma de dicionário e índice: 
df.groupby('STNAME').groups

{'Alabama': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67], 'Alaska': [69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97], 'Arizona': [99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113], 'Arkansas': [115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189], 'California': [191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212

In [ ]:
# 99% das vezes, é usado groupby em uma ou mais colunas. Além disso, também podemos fornecer uma função para o groupby e usá-la para segmentar os dados.

# Vamos supor que temos um grande trabalho em lote com muito processamento e queremos trabalhar apenas em cerca de um terço dos estados em um determinado momento. Nesse caso, poderíamos criar alguma função que retornasse um número entre zero e dois com base no primeiro caractere do nome do estado. Sendo assim, podemos dizer ao groupby que use essa função para dividir nosso DataFrame. 

# ATENÇÃO: É importante observar que, para fazer isso, precisamos definir o índice do DataFrame como a coluna pela qual você deseja agrupar primeiro.

# Criaremos uma nova função chamada definir_numero_lote(), e se o primeiro número, o parâmetro for até M, retornaremos um zero, se for > M até Q, retornaremos um, e caso contrário, retornaremo dois. Em seguida, passaremos essa função para o DataFrame.

# Coluna pela qual vamos agrupar
df = df.set_index('STNAME')

def definir_numero_lote(item: str):
    if item[0] < 'M': # Isso é do Python, por exemplo 'A' < 'M', 'B' < 'M'...
        return 0
    
    if item[0] < 'Q': 
        return 1
    
    return 2

# O DataFrame deve ser agrupado de acordo com o número do lote. Vamos percorrer cada grupo de lotes
for grupo, frame in df.groupby(definir_numero_lote): 
    print(f'Temos {str(len(frame))} registros no grupo, Grupo: {grupo} para processamento')

Temos 1177 registros no grupo, Grupo: 0 para processamento
Temos 1134 registros no grupo, Grupo: 1 para processamento
Temos 831 registros no grupo, Grupo: 2 para processamento


Perceba que dessa vez, não passamos o nome de uma coluna para o groupby(). Em vez disso, definimos o índice do DataFrame como 'STNAME' e, se nenhum identificador de coluna for passado, o groupby usará automaticamente esse índice. 

Função aplicada ao índice: <br>
df.groupby(func)

E se não quisermos definir um índice para isso? 

In [ ]:
# Se não quisermos definir um índice, a forma mais indicada é criar uma nova coluna para separação dos dados usando a função map():

df['lote'] = df['STNAME'].map(definir_numero_lote)

for grupo, frame in df.groupby('lote'):
    print(f'Temos {len(frame)} registros no grupo {grupo}')

Temos 1177 registros no grupo 0
Temos 1134 registros no grupo 1
Temos 831 registros no grupo 2


In [60]:
# Ou até mesmo criar uma nova coluna:
for grupo, frame in df.groupby(df['STNAME'].map(definir_numero_lote)):
    print(f'Temos {len(frame)} registros no grupo {grupo}')

Temos 1177 registros no grupo 0
Temos 1134 registros no grupo 1
Temos 831 registros no grupo 2


ATENÇÃO: Podemos utilizar o groupby() com mais de uma coluna, basta passar uma lista de colunas ao invés de uma. 

df.groupby([coluna1, coluna2])

In [67]:
# Vamos supor o DataFrame 
df = pd.DataFrame({
    'Estado': ['SP', 'SP', 'RJ', 'RJ', 'SP'],
    'Cidade': ['Campinas', 'São Paulo', 'Niterói', 'Rio de Janeiro', 'Campinas'],
    'População': [1, 12, 0.5, 6, 1.2]
})

# Agrupando por Estado e Cidade:
df.groupby(['Estado', 'Cidade'])

# Calculando alguma agregação
df.groupby(['Estado', 'Cidade'])['População'].mean()

# Iterando sobre os grupos
for (estado, cidade), frame in df.groupby(['Estado', 'Cidade']):
    print(estado, cidade)
    display(frame)

RJ Niterói


,Estado,Cidade,População
2,RJ,Niterói,0.5


RJ Rio de Janeiro


,Estado,Cidade,População
3,RJ,Rio de Janeiro,6.0


SP Campinas


,Estado,Cidade,População
0,SP,Campinas,1.0
4,SP,Campinas,1.2


SP São Paulo


,Estado,Cidade,População
1,SP,São Paulo,12.0


Vamos ver mais um exemplo de como podemos agrupar os dados

In [69]:
# Vamos dar mais uma olhada em um exemplo de como podemos agrupar dados. 

df = pd.read_csv('datasets/listings.csv')
df.head()

,id,listing_url,scrape_id,last_scraped,name,summary,space,description,experiences_offered,neighborhood_overview,...,review_scores_value,requires_license,license,jurisdiction_names,instant_bookable,cancellation_policy,require_guest_profile_picture,require_guest_phone_verification,calculated_host_listings_count,reviews_per_month
0,12147973,https://www.airbnb.com/rooms/12147973,20160906204935,2016-09-07,Sunny Bungalow in the City,"Cozy, sunny, family home. Master bedroom high...",The house has an open and cozy feel at the sam...,"Cozy, sunny, family home. Master bedroom high...",none,"Roslindale is quiet, convenient and friendly. ...",...,NaN,f,NaN,NaN,f,moderate,f,f,1,NaN
1,3075044,https://www.airbnb.com/rooms/3075044,20160906204935,2016-09-07,Charming room in pet friendly apt,Charming and quiet room in a second floor 1910...,Small but cozy and quite room with a full size...,Charming and quiet room in a second floor 1910...,none,"The room is in Roslindale, a diverse and prima...",...,9.0,f,NaN,NaN,t,moderate,f,f,1,1.30
2,6976,https://www.airbnb.com/rooms/6976,20160906204935,2016-09-07,Mexican Folk Art Haven in Boston,"Come stay with a friendly, middle-aged guy in ...","Come stay with a friendly, middle-aged guy in ...","Come stay with a friendly, middle-aged guy in ...",none,The LOCATION: Roslindale is a safe and diverse...,...,10.0,f,NaN,NaN,f,moderate,t,f,1,0.47
3,1436513,https://www.airbnb.com/rooms/1436513,20160906204935,2016-09-07,Spacious Sunny Bedroom Suite in Historic Home,Come experience the comforts of home away from...,Most places you find in Boston are small howev...,Come experience the comforts of home away from...,none,Roslindale is a lovely little neighborhood loc...,...,10.0,f,NaN,NaN,f,moderate,f,f,1,1.00
4,7651065,https://www.airbnb.com/rooms/7651065,20160906204935,2016-09-07,Come Home to Boston,"My comfy, clean and relaxing home is one block...","Clean, attractive, private room, one block fro...","My comfy, clean and relaxing home is one block...",none,"I love the proximity to downtown, the neighbor...",...,10.0,f,NaN,NaN,f,flexible,f,f,1,2.25


Nesse conjunto de dados, temos duas colunas de interesse: cancellation_policy (politica de cancelamento) e review_scores_value (valor das avaliações)

In [70]:
# Como podemos agrupar essas duas colunas? 
# Uma primeira abordagem pode ser promovê-los a um índice múltiplo e depois chamar groupby.

df = df.set_index(['cancellation_policy', 'review_scores_value'])

# Quando temos múltiplos índices, precisamos passar os níveis pelos quais estamos interessados em agrupar, uma vez que, por padrão, o groupby() 
for grupo, frame in df.groupby(level = (0, 1)):
    print(grupo)

('flexible', 2.0)
('flexible', 4.0)
('flexible', 5.0)
('flexible', 6.0)
('flexible', 7.0)
('flexible', 8.0)
('flexible', 9.0)
('flexible', 10.0)
('moderate', 2.0)
('moderate', 4.0)
('moderate', 6.0)
('moderate', 7.0)
('moderate', 8.0)
('moderate', 9.0)
('moderate', 10.0)
('strict', 2.0)
('strict', 3.0)
('strict', 4.0)
('strict', 5.0)
('strict', 6.0)
('strict', 7.0)
('strict', 8.0)
('strict', 9.0)
('strict', 10.0)
('super_strict_30', 6.0)
('super_strict_30', 7.0)
('super_strict_30', 8.0)
('super_strict_30', 9.0)
('super_strict_30', 10.0)


In [71]:
# E se quiséssemos agrupar pela política de cancelamento e pelas notas de avaliação, mas separar todas as dezenas das menores de 10 anos?
# Nesse caso, poderíamos usar uma função para gerenciar os agrupamentos. 

def agrupando(item: tuple):
    # Verificaremos a parte do review_scores_value
    if item[1] == 10.0:
        return (item[0], '10.0')
    
    else: 
        return (item[0], 'não é 10.0')

for grupo, frame in df.groupby(by = agrupando):
    print(grupo)

('flexible', '10.0')
('flexible', 'não é 10.0')
('moderate', '10.0')
('moderate', 'não é 10.0')
('strict', '10.0')
('strict', 'não é 10.0')
('super_strict_30', '10.0')
('super_strict_30', 'não é 10.0')
